In [5]:
"""
CGNT Stage 2 — Residual GCN
==============================
Takes the frozen Stage 1 transformer and trains the GCN (v3 architecture)
on RESIDUALS rather than raw attack deltas.

Pipeline for each training sample:
  1. Pick a random benign window [t-48 : t-1] from hourly_dataset
  2. Get transformer prediction for hour t  (frozen, no grad)
  3. Inject attack vector at hour t:
       observed[t] = benign[t] + attack_delta
  4. Compute residual = observed[t] - predicted[t]
  5. Feed residual + H_row + type_embed into 3-layer GATv2
  6. Classify each of 54 sensors: CLEAN / PRIMARY / SECONDARY

Training data : cgnt_dataset_v2.csv  (10,000 attack samples)
                hourly_dataset.xlsx  (benign windows for context)
Frozen model  : transformer_best.pt + transformer_norm.npz

Outputs:
  residual_gcn_best.pt        — best GCN weights
  stage2_test_results.npz     — predictions + labels for analysis

Author: RAAGAVAN CGNT Research
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import pandas as pd
from torch_geometric.nn import GATv2Conv
from torch_geometric.data import Data, DataLoader
from sklearn.metrics import f1_score, classification_report
from sklearn.model_selection import train_test_split
import math
import warnings
warnings.filterwarnings('ignore')

# ─────────────────────────────────────────────
# 0.  CONFIG
# ─────────────────────────────────────────────
HOURLY_PATH   = "hourly_dataset.xlsx"
DATASET_PATH  = "cgnt_dataset_v3.csv"
H_MATRIX_PATH = "H_true_ieee14.npy"
TRANSFORMER_PT   = "transformer_best.pt"
TRANSFORMER_NORM = "transformer_norm.npz"

WINDOW       = 48
BASE_MVA     = 100.0   # attack deltas are in pu; multiply by this to convert to MW
BATCH_SIZE   = 64
EPOCHS       = 200
LR           = 1e-3
WARMUP_EPOCHS = 5
HIDDEN_DIM   = 64
NUM_HEADS    = 4
DROPOUT      = 0.3
WEIGHT_DECAY = 3e-4
EARLY_STOP   = 30
SENSOR_EMBED_DIM = 8
RANDOM_SEED  = 42
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
print(f"Device: {DEVICE}")


# ─────────────────────────────────────────────
# 1.  TEMPORAL TRANSFORMER  (frozen, copy from Stage 1)
# ─────────────────────────────────────────────
import os, sys

class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=200, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(dropout)
        pe = torch.zeros(max_len, d_model)
        pos = torch.arange(0, max_len).unsqueeze(1).float()
        div = torch.exp(torch.arange(0, d_model, 2).float()
                        * (-math.log(10000.0) / d_model))
        pe[:, 0::2] = torch.sin(pos * div)
        pe[:, 1::2] = torch.cos(pos * div)
        self.register_buffer('pe', pe.unsqueeze(0))

    def forward(self, x):
        return self.dropout(x + self.pe[:, :x.size(1)])


class TemporalTransformer(nn.Module):
    def __init__(self, d_model=54, nhead=6, num_layers=2,
                 lstm_hidden=128, dropout=0.1):
        super().__init__()
        self.pos_enc = PositionalEncoding(d_model, dropout=dropout)
        enc_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=nhead,
            dim_feedforward=d_model * 4,
            dropout=dropout, batch_first=True)
        self.transformer = nn.TransformerEncoder(enc_layer,
                                                  num_layers=num_layers)
        self.lstm1 = nn.LSTM(d_model, lstm_hidden, batch_first=True)
        self.lstm2 = nn.LSTM(lstm_hidden, lstm_hidden, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(lstm_hidden, lstm_hidden),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(lstm_hidden, d_model))

    def forward(self, x):
        x = self.pos_enc(x)
        x = self.transformer(x)
        x, _ = self.lstm1(x)
        x, _ = self.lstm2(x)
        x = x[:, -1, :]
        return self.head(x)


def load_frozen_transformer(pt_path, norm_path, device):
    norm = np.load(norm_path)
    mu   = norm['mu'].astype(np.float32)   # (1, 54)
    sig  = norm['sig'].astype(np.float32)  # (1, 54)

    model = TemporalTransformer().to(device)
    model.load_state_dict(torch.load(pt_path, map_location=device))
    model.eval()
    for p in model.parameters():
        p.requires_grad = False

    print(f"Transformer loaded from {pt_path}  [FROZEN]")
    return model, mu, sig


# ─────────────────────────────────────────────
# 2.  IEEE-14 TOPOLOGY
# ─────────────────────────────────────────────
LINES = [
    (1,2),(1,5),(2,3),(2,4),(2,5),(3,4),(4,5),(4,7),(4,9),
    (5,6),(6,11),(6,12),(6,13),(7,8),(7,9),(9,10),(9,14),
    (10,11),(12,13),(13,14)
]
SENSOR_TYPE_IDS = np.array([0]*20 + [1]*20 + [2]*14, dtype=np.int64)
NUM_SENSOR_TYPES = 3


def build_sensor_graph():
    edges = set()
    def add_edge(i, j):
        if i != j: edges.add((min(i,j), max(i,j)))
    for line_idx, (f_bus, t_bus) in enumerate(LINES):
        fwd  = line_idx;      rev  = line_idx + 20
        fbus = 40+(f_bus-1);  tbus = 40+(t_bus-1)
        add_edge(fwd, rev);   add_edge(fwd, fbus);  add_edge(fwd, tbus)
        add_edge(rev, fbus);  add_edge(rev, tbus);  add_edge(fbus, tbus)
    src, dst = [], []
    for i, j in edges:
        src.extend([i, j]); dst.extend([j, i])
    edge_index = torch.tensor([src, dst], dtype=torch.long)
    print(f"Sensor graph: 54 nodes, {len(edges)} undirected edges")
    return edge_index


def load_H_features(path):
    H = np.load(path).astype(np.float32)
    H_norm = (H - H.mean(0)) / (H.std(0) + 1e-8)
    print(f"H matrix: {H.shape}, norm range [{H_norm.min():.3f}, {H_norm.max():.3f}]")
    return H_norm


# ─────────────────────────────────────────────
# 3.  RESIDUAL DATASET BUILDER
# ─────────────────────────────────────────────

def build_residual_dataset(transformer, mu, sig, device):
    """
    For every attack sample in cgnt_dataset_v2:
      - randomly pick a benign window from hourly data
      - get transformer prediction (frozen)
      - residual = (benign[t] + attack_delta) - predicted[t]
      - label = causal labels from dataset

    Returns:
      residuals : (N, 54)  float32
      labels    : (N, 54)  int64
      k_values  : (N,)
    """
    print("\nLoading benign hourly data...")
    df_h = pd.read_excel(HOURLY_PATH)
    line_cols = [f'line_{i}' for i in range(1, 41)]
    cons_cols = [f'Cons_{i}'  for i in range(1, 15)]
    benign_mw   = df_h[line_cols + cons_cols].values.astype(np.float32)  # (8760, 54) MW

    # Normalize benign using Stage 1 normalization (fit on MW data)
    benign_norm = (benign_mw - mu) / sig           # (8760, 54)

    print("Loading attack dataset...")
    df_a = pd.read_csv(DATASET_PATH)
    noisy_cols = [f'noisy_s{i}' for i in range(1, 55)]
    label_cols = [f'label_s{i}'  for i in range(1, 55)]
    clean_cols = [f's{i}'         for i in range(1, 55)]

    # Attack delta = clean signal (what was injected, in pu)
    # We use the clean attack delta directly (not noisy)
    # because the temporal transformer removes the benign baseline
    # and the residual IS the attack signal
    attack_deltas = df_a[clean_cols].values.astype(np.float32)   # (N, 54)
    labels        = df_a[label_cols].values.astype(np.int64)      # (N, 54)
    k_values      = df_a['k_value'].values

    N = len(attack_deltas)
    rng = np.random.default_rng(RANDOM_SEED)

    # Random window indices (ensure we have full 48h before t)
    # Use hours 48..8759 to always have a full window available
    t_indices = rng.integers(WINDOW, len(benign_mw), size=N)

    print(f"Computing residuals for {N} samples (frozen transformer)...")
    residuals = np.zeros((N, 54), dtype=np.float32)

    transformer.eval()
    bs = 256
    with torch.no_grad():
        for start in range(0, N, bs):
            end   = min(start + bs, N)
            batch_t = t_indices[start:end]

            # Build context windows: (batch, 48, 54)
            contexts = np.stack([benign_norm[t-WINDOW:t]
                                  for t in batch_t])  # (batch, 48, 54)
            ctx_tensor = torch.tensor(contexts, dtype=torch.float32).to(device)

            # Transformer prediction (normalized)
            pred_norm = transformer(ctx_tensor).cpu().numpy()  # (batch, 54)

            # Benign values at time t (normalized)
            benign_t_norm = np.stack([benign_norm[t] for t in batch_t])

            # Attack deltas are in pu — convert to MW to match benign scale
            delta_mw   = attack_deltas[start:end] * BASE_MVA          # pu -> MW
            delta_norm = delta_mw / sig                                # normalize

            # Clip to prevent sensor 47 (sig≈0, bus_7 always 0) from exploding
            delta_norm = np.clip(delta_norm, -20.0, 20.0)

            # Observed (normalized) = benign_t_norm + delta_norm
            observed_norm = benign_t_norm + delta_norm

            # Residual = observed - predicted  (in normalized space)
            residuals[start:end] = observed_norm - pred_norm

            if (start // bs) % 10 == 0:
                print(f"  {end}/{N} samples processed...")

    print(f"Residuals computed.")
    print(f"  Residual range: [{residuals.min():.4f}, {residuals.max():.4f}]")
    print(f"  Mean abs residual (attacked sensors): "
          f"{np.abs(residuals[labels > 0]).mean():.4f}")
    print(f"  Mean abs residual (clean sensors):    "
          f"{np.abs(residuals[labels == 0]).mean():.4f}")

    return residuals, labels, k_values


# ─────────────────────────────────────────────
# 4.  GCN MODEL  (identical to v3 baseline)
# ─────────────────────────────────────────────

class ResidualGCN(nn.Module):
    """
    3-layer GATv2 operating on transformer residuals.
    Identical architecture to GCN v3 baseline — the only change
    is that node input[0] is now the RESIDUAL not the raw attack delta.
    This is the key improvement: cleaner, more informative input signal.
    """

    def __init__(self, h_dim=13, embed_dim=8, hidden_dim=64,
                 num_heads=4, num_classes=3, dropout=0.3):
        super().__init__()
        self.dropout = dropout
        in_dim = 1 + h_dim + embed_dim   # residual(1) + H_row(13) + embed(8) = 22

        self.type_embed  = nn.Embedding(NUM_SENSOR_TYPES, embed_dim)
        self.input_proj  = nn.Linear(in_dim, hidden_dim)

        self.conv1 = GATv2Conv(in_dim, hidden_dim, heads=num_heads,
                               dropout=dropout, concat=True)
        self.conv2 = GATv2Conv(hidden_dim*num_heads, hidden_dim,
                               heads=num_heads, dropout=dropout, concat=True)
        self.conv3 = GATv2Conv(hidden_dim*num_heads, hidden_dim,
                               heads=1, dropout=dropout, concat=False)

        self.bn1 = nn.BatchNorm1d(hidden_dim * num_heads)
        self.bn2 = nn.BatchNorm1d(hidden_dim * num_heads)
        self.bn3 = nn.BatchNorm1d(hidden_dim)

        self.classifier = nn.Sequential(
            nn.Linear(hidden_dim * 2, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, num_classes)
        )

    def forward(self, x, edge_index, sensor_type):
        type_emb = self.type_embed(sensor_type)
        x_full   = torch.cat([x, type_emb], dim=1)        # (N, 22)
        residual = F.relu(self.input_proj(x_full))         # (N, 64)

        h = F.elu(self.bn1(self.conv1(x_full, edge_index)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.elu(self.bn2(self.conv2(h, edge_index)))
        h = F.dropout(h, p=self.dropout, training=self.training)
        h = F.elu(self.bn3(self.conv3(h, edge_index)))

        h = torch.cat([h, residual], dim=1)
        return self.classifier(h)


# ─────────────────────────────────────────────
# 5.  DATASET → PyG DATA LIST
# ─────────────────────────────────────────────

def make_pyg_data_list(residuals, labels, edge_index, H_features):
    sensor_type_ids = torch.tensor(SENSOR_TYPE_IDS, dtype=torch.long)
    H_feat = torch.tensor(H_features, dtype=torch.float32)
    data_list = []
    for i in range(len(residuals)):
        r = torch.tensor(residuals[i], dtype=torch.float32).unsqueeze(1)
        x = torch.cat([r, H_feat], dim=1)   # (54, 14)
        y = torch.tensor(labels[i], dtype=torch.long)
        data_list.append(Data(x=x, edge_index=edge_index,
                              y=y, sensor_type=sensor_type_ids))
    return data_list


def compute_class_weights(Y):
    flat    = Y.ravel()
    counts  = np.bincount(flat, minlength=3).astype(float)
    weights = 1.0 / (np.log1p(counts) + 1e-8)
    weights = weights / weights.sum() * 3
    w = torch.tensor(weights, dtype=torch.float32)
    print(f"Class counts  — CLEAN:{counts[0]:.0f}  "
          f"PRIMARY:{counts[1]:.0f}  SECONDARY:{counts[2]:.0f}")
    print(f"Class weights — CLEAN:{w[0]:.3f}  "
          f"PRIMARY:{w[1]:.3f}  SECONDARY:{w[2]:.3f}")
    return w


# ─────────────────────────────────────────────
# 6.  LR SCHEDULE
# ─────────────────────────────────────────────

def get_lr(epoch, warmup, total, base_lr):
    if epoch < warmup:
        return base_lr * (epoch + 1) / warmup
    progress = (epoch - warmup) / max(total - warmup, 1)
    return base_lr * 0.5 * (1 + math.cos(math.pi * progress))


# ─────────────────────────────────────────────
# 7.  TRAIN / EVAL
# ─────────────────────────────────────────────

def train_epoch(model, loader, optimizer, criterion, device):
    model.train()
    total = 0.0
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.sensor_type)
        loss = criterion(out, batch.y)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total += loss.item()
    return total / len(loader)


@torch.no_grad()
def evaluate(model, loader, criterion, device):
    model.eval()
    total, preds, labels = 0.0, [], []
    for batch in loader:
        batch = batch.to(device)
        out   = model(batch.x, batch.edge_index, batch.sensor_type)
        total += criterion(out, batch.y).item()
        preds.append(out.argmax(dim=1).cpu().numpy())
        labels.append(batch.y.cpu().numpy())
    return (total / len(loader),
            np.concatenate(preds),
            np.concatenate(labels))


def compute_racc(preds, labels, n=54):
    m = len(preds) // n
    return (preds.reshape(m, n) == labels.reshape(m, n)).all(axis=1).mean()


def print_metrics(preds, labels, name):
    print(f"\n{'='*52}\n  {name}\n{'='*52}")
    print(classification_report(labels, preds,
          target_names=['CLEAN','PRIMARY','SECONDARY'],
          digits=4, zero_division=0))
    racc = compute_racc(preds, labels)
    print(f"  RACC : {racc:.4f}  ({100*racc:.2f}%)")
    print(f"  XTM  : 0.9299  (92.99%)")
    print(f"  Gap  : {0.9299-racc:+.4f}")
    print(f"{'='*52}")
    return racc


# ─────────────────────────────────────────────
# 8.  MAIN
# ─────────────────────────────────────────────

def main():
    print("\n" + "="*60)
    print("  CGNT Stage 2 — Residual GCN")
    print("  GCN trained on transformer residuals")
    print("="*60)

    # Load frozen transformer
    transformer, mu, sig = load_frozen_transformer(
        TRANSFORMER_PT, TRANSFORMER_NORM, DEVICE)

    # Build graph and H features
    edge_index = build_sensor_graph()
    H_features = load_H_features(H_MATRIX_PATH)

    # Build residual dataset
    residuals, labels, k_values = build_residual_dataset(
        transformer, mu, sig, DEVICE)

    # 70/15/15 split stratified by k
    idx = np.arange(len(residuals))
    idx_train, idx_temp = train_test_split(idx, test_size=0.30,
                          random_state=RANDOM_SEED, stratify=k_values)
    idx_val, idx_test   = train_test_split(idx_temp, test_size=0.50,
                          random_state=RANDOM_SEED,
                          stratify=k_values[idx_temp])
    print(f"\nSplit — Train:{len(idx_train)}  "
          f"Val:{len(idx_val)}  Test:{len(idx_test)}")

    ei_cpu = edge_index.cpu()
    train_data = make_pyg_data_list(residuals[idx_train],
                                    labels[idx_train], ei_cpu, H_features)
    val_data   = make_pyg_data_list(residuals[idx_val],
                                    labels[idx_val],   ei_cpu, H_features)
    test_data  = make_pyg_data_list(residuals[idx_test],
                                    labels[idx_test],  ei_cpu, H_features)

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True)
    val_loader   = DataLoader(val_data,   batch_size=BATCH_SIZE, shuffle=False)
    test_loader  = DataLoader(test_data,  batch_size=BATCH_SIZE, shuffle=False)

    class_weights = compute_class_weights(labels[idx_train]).to(DEVICE)
    criterion     = nn.CrossEntropyLoss(weight=class_weights)

    model = ResidualGCN(
        h_dim=13, embed_dim=SENSOR_EMBED_DIM,
        hidden_dim=HIDDEN_DIM, num_heads=NUM_HEADS,
        num_classes=3, dropout=DROPOUT
    ).to(DEVICE)
    print(f"\nGCN parameters: {sum(p.numel() for p in model.parameters()):,}")

    optimizer = torch.optim.Adam(model.parameters(),
                                  lr=LR, weight_decay=WEIGHT_DECAY)

    # Training
    print(f"\nTraining for up to {EPOCHS} epochs...")
    print(f"{'Epoch':>6} {'LR':>10} {'Train Loss':>12} "
          f"{'Val Loss':>10} {'Val RACC':>10}")
    print("-" * 55)

    best_val_loss, best_epoch, patience = float('inf'), 0, 0

    for epoch in range(1, EPOCHS + 1):
        lr = get_lr(epoch-1, WARMUP_EPOCHS, EPOCHS, LR)
        for pg in optimizer.param_groups: pg['lr'] = lr

        train_loss = train_epoch(model, train_loader,
                                  optimizer, criterion, DEVICE)
        val_loss, val_preds, val_labels = evaluate(
            model, val_loader, criterion, DEVICE)
        val_racc = compute_racc(val_preds, val_labels)

        if epoch % 5 == 0 or epoch == 1:
            print(f"{epoch:>6} {lr:>10.6f} {train_loss:>12.4f} "
                  f"{val_loss:>10.4f} {val_racc:>10.4f}")

        if val_loss < best_val_loss:
            best_val_loss = val_loss; best_epoch = epoch; patience = 0
            torch.save(model.state_dict(), "residual_gcn_best.pt")
        else:
            patience += 1
            if patience >= EARLY_STOP:
                print(f"\nEarly stopping at epoch {epoch} "
                      f"(best: epoch {best_epoch})")
                break

    # Final evaluation
    print(f"\nLoading best model from epoch {best_epoch}...")
    model.load_state_dict(
        torch.load("residual_gcn_best.pt", map_location=DEVICE))

    _, val_preds,  val_labels  = evaluate(model, val_loader,
                                           criterion, DEVICE)
    _, test_preds, test_labels = evaluate(model, test_loader,
                                           criterion, DEVICE)

    print_metrics(val_preds,  val_labels,  "VALIDATION")
    test_racc = print_metrics(test_preds, test_labels, "TEST")

    # Per-k breakdown
    print("\n=== PERFORMANCE BY k_value ===")
    k_test  = k_values[idx_test]
    p2d = test_preds.reshape(-1, 54)
    l2d = test_labels.reshape(-1, 54)
    for k in sorted(np.unique(k_test)):
        mask = k_test == k
        p = p2d[mask].ravel(); l = l2d[mask].ravel()
        racc  = compute_racc(p2d[mask].ravel(), l2d[mask].ravel())
        f1_p  = f1_score(l, p, labels=[1], average='macro', zero_division=0)
        f1_s  = f1_score(l, p, labels=[2], average='macro', zero_division=0)
        print(f"  k={k}: RACC={racc:.4f}  "
              f"PRIMARY_F1={f1_p:.4f}  SECONDARY_F1={f1_s:.4f}  "
              f"(n={mask.sum()})")

    # Save test results for Stage 3 analysis
    np.savez("stage2_test_results.npz",
             preds=test_preds, labels=test_labels,
             k_values=k_test, residuals=residuals[idx_test])
    print("\nSaved: stage2_test_results.npz")
    print("Saved: residual_gcn_best.pt")

    # Comparison table
    print("\n=== STAGE COMPARISON ===")
    print(f"{'Stage':<30} {'RACC':>8} {'vs XTM':>10}")
    print("-" * 50)
    print(f"{'GCN v3 (raw deltas)':<30} {'0.8867':>8} {'-0.0432':>10}")
    print(f"{'Stage 2 (residuals)':<30} {test_racc:>8.4f} "
          f"{test_racc-0.9299:>+10.4f}")
    print(f"{'XTM baseline':<30} {'0.9299':>8} {'---':>10}")

    print("\n=== DECISION ===")
    if test_racc >= 0.93:
        print(">> RACC >= 0.93: BEATS XTM with Stage 2 alone.")
        print(">> Stage 3 (CIE) will further improve causal attribution.")
    elif test_racc >= 0.90:
        print(">> RACC >= 0.90: Very close to XTM. Stage 3 should push past it.")
    elif test_racc >= 0.87:
        print(">> RACC >= 0.87: Residuals helped vs raw deltas.")
        print(">> Stage 3 CIE needed to close remaining gap.")
    else:
        print(">> Check residual magnitudes — attack scale may need adjustment.")


if __name__ == '__main__':
    main()

Device: cuda

  CGNT Stage 2 — Residual GCN
  GCN trained on transformer residuals
Transformer loaded from transformer_best.pt  [FROZEN]
Sensor graph: 54 nodes, 120 undirected edges
H matrix: (54, 13), norm range [-3.674, 5.453]

Loading benign hourly data...
Loading attack dataset...
Computing residuals for 10000 samples (frozen transformer)...
  256/10000 samples processed...
  2816/10000 samples processed...
  5376/10000 samples processed...
  7936/10000 samples processed...
Residuals computed.
  Residual range: [-20.7546, 20.7548]
  Mean abs residual (attacked sensors): 8.8857
  Mean abs residual (clean sensors):    0.0596

Split — Train:7000  Val:1500  Test:1500
Class counts  — CLEAN:268681  PRIMARY:80266  SECONDARY:29053
Class weights — CLEAN:0.903  PRIMARY:0.999  SECONDARY:1.098

GCN parameters: 188,507

Training for up to 200 epochs...
 Epoch         LR   Train Loss   Val Loss   Val RACC
-------------------------------------------------------
     1   0.000200       0.6905     